## BLC data Ingestion Framework
NoteBook: Bronze_ingestion_rearc

| Author | Date | Change Record | Description |
| ----------- | ----------- | ----------- | -----------
|Hemant kaushik	|05-Nov-2026|	-	| Notebook for ingesting BLC data
 



In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import col,trim,current_timestamp,input_file_name,when,from_json
from pyspark.sql.types import StructType,StructField,StringType

In [0]:
base_path = "/Volumes/workspace/hemant_poc/volume_dev/sourcefiles/"
table_prefix = "workspace.bronze.bronze_pr_"

In [0]:
series_schema = StructType([
    StructField("series_id", StringType(), False),
    StructField("sector_code", StringType(), True),
    StructField("class_code", StringType(), True),
    StructField("measure_code", StringType(), True),
    StructField("duration_code", StringType(), True),
    StructField("seasonal", StringType(), True),
    StructField("base_year", StringType(), True),
    StructField("footnote_codes", StringType(), True),
    StructField("begin_year", StringType(), True),
    StructField("begin_period", StringType(), True),
    StructField("end_year", StringType(), True),
    StructField("end_period", StringType(), True),
    StructField("_corrupt_record", StringType(), True)
])

sector_schema = StructType([
    StructField("sector_code", StringType(), True),
    StructField("sector_name", StringType(), True),	
    StructField("display_level", StringType(), True),
    StructField("selectable", StringType(), True),
    StructField("sort_sequence", StringType(), True),
    StructField("_corrupt_record", StringType(), True)
])

measure_schema = StructType([
    StructField("measure_code", StringType(), True),
    StructField("measure_text", StringType(), True),	
    StructField("display_level", StringType(), True),
    StructField("selectable", StringType(), True),
    StructField("sort_sequence", StringType(), True),
    StructField("_corrupt_record", StringType(), True)
])


duration_schema = StructType([
    StructField("duration_code", StringType(), True),
    StructField("duration_text", StringType(), True),	
    StructField("display_level", StringType(), True),
    StructField("selectable", StringType(), True),
    StructField("sort_sequence", StringType(), True),
    StructField("_corrupt_record", StringType(), True)
])


footnote_schema = StructType([
    StructField("footnote_code", StringType(), True),
    StructField("footnote_text", StringType(), True),
    StructField("_corrupt_record", StringType(), True)
])


period_schema = StructType([
    StructField("period", StringType(), True),
    StructField("period_abbr", StringType(), True),	
    StructField("period_name", StringType(), True),
    StructField("_corrupt_record", StringType(), True)
])


seasonal_schema = StructType([
    StructField("seasonal_code", StringType(), True),
    StructField("seasonal_text", StringType(), True),
    StructField("_corrupt_record", StringType(), True)
])


class_schema = StructType([
    StructField("class_code", StringType(), True),
    StructField("class_text", StringType(), True),
    StructField("display_level", StringType(), True),
    StructField("selectable", StringType(), True),
    StructField("sort_sequence", StringType(), True),
    StructField("_corrupt_record", StringType(), True)
])

data_schema = StructType([
    StructField("series_id", StringType(), False),
    StructField("year", StringType(), True),
    StructField("period", StringType(), True),
    StructField("value", StringType(), True),
    StructField("footnote_codes", StringType(), True),
    StructField("_corrupt_record", StringType(), True)
])



In [0]:
source_config = [
        {
        "table_name": f"{table_prefix}series",
        "file_name": "pr.series",
        "file_format": "csv",
        "schema": series_schema,
        "delimiter": "\t",
        "expectations": {
            "series_id_not_null": "series_id IS NOT NULL"
            }
        },
        {
        "table_name": f"{table_prefix}sector",
        "file_name": "pr.sector",
        "file_format": "csv",
        "schema": sector_schema,
        "delimiter": "\t",
        "expectations": {
            "sector_code_not_null": "sector_code IS NOT NULL"
            }            
        },
        {
        "table_name": f"{table_prefix}measure",
        "file_name": "pr.measure",
        "file_format": "csv",
        "schema": measure_schema,
        "delimiter": "\t",
        "expectations": {
            "measure_code_not_null": "measure_code IS NOT NULL"
             }
        },
        {
        "table_name": f"{table_prefix}duration",
        "file_name": "pr.duration",
        "file_format": "csv",
        "schema": duration_schema,
        "delimiter": "\t",
        "expectations": {
        "duration_code_not_null": "duration_code IS NOT NULL"
            }
        },
        {
        "table_name": f"{table_prefix}footnote",
        "file_name": "pr.footnote",
        "file_format": "csv",
        "schema": footnote_schema,
        "delimiter": "\t",
        "expectations": {
        "footnote_code_not_null": "footnote_code IS NOT NULL"
            }
        },
        {
        "table_name": f"{table_prefix}period",
        "file_name": "pr.period",
        "file_format": "csv",
        "schema": period_schema,
        "delimiter": "\t",
        "expectations": {
        "period_not_null": "period IS NOT NULL"
            }
        },
        {
        "table_name": f"{table_prefix}seasonal",
        "file_name": "pr.seasonal",
        "file_format": "csv",
        "schema": seasonal_schema,
        "delimiter": "\t",
        "expectations": {
        "seasonal_code_not_null": "seasonal_code IS NOT NULL"
            }
        },
        {
        "table_name": f"{table_prefix}class",
        "file_name": "pr.class",
        "file_format": "csv",
        "schema": class_schema,
        "delimiter": "\t",
        "expectations": {
        "class_code_not_null": "class_code IS NOT NULL"
            }
        },
        {
        "table_name": f"{table_prefix}population",
        "file_name": "data*",
        "file_format": "json",
        "schema": None,
        "delimiter": None,
        "expectations": None
        },
        {
        "table_name": f"{table_prefix}data",
        "file_name": "pr.data*",
        "file_format": "csv",
        "schema": data_schema,
        "delimiter": "\t",
        "expectations": {
        "series_id_not_null": "series_id IS NOT NULL",
        "year_not_null": "year IS NOT NULL",
        "period_not_null": "period IS NOT NULL",
        "value_not_null": "value IS NOT NULL"
            }
        }
]

In [0]:
def create_bronze_table(config):

    table_name = config["table_name"]
    file_name = config["file_name"]
    file_format = config["file_format"]
    schema = config["schema"]
    delimiter = config.get("delimiter", "\t")
    expectations = config.get("expectations", {})

    @dp.materialized_view(
        name=table_name,
        comment=f"Raw data has been loaded from {file_name}"
    )
    #@dp.expect_all_or_drop(expectations)
    def bronze_table():
        # -------------------------
        # CSV / TSV / BLS files
        # -------------------------
        if file_format == "csv":
            df = (
                spark.read
                .option("header", True)
                .option("sep", delimiter)
                .option("mode", "PERMISSIVE")
                .option("columnNameOfCorruptRecord", "_corrupt_record")
                .schema(schema)
                .csv(f"{base_path}/{file_name}")
            )
           # Trim all string columns dynamically
            for field in df.schema.fields:
                if field.dataType.simpleString() == "string":
                    df = df.withColumn(
                         field.name,
                         when(trim(col(field.name)) == "",None).otherwise(trim(col(field.name)))
                        )
        # -------------------------
        # JSON files
        # -------------------------
        elif file_format == "json":

            df = (
                spark.read
                .format("text")
                .option("wholetext", True)
                .load(f"{base_path}/{file_name}")
                .select(
                    col("value").alias("raw_json"),
                    col("_metadata.file_path").alias("_source_file")
                )
            )  

        return (
            df
            .withColumn("_source_file", col("_metadata.file_path"))
            .withColumn("_ingested_at", current_timestamp())
        )



In [0]:
for config in source_config:
    create_bronze_table(config)
